# LeJEPA CUDA Runner for Colab

Use **Runtime > Change runtime type > GPU** before running. This notebook runs the existing repo with CUDA PyTorch in Colab, downloads STL-10 if needed, trains `model.run`, and optionally runs `model.probe`.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

# Option A: put your repo on GitHub and set REPO_URL.
# Option B: upload/copy the repo to PROJECT_DIR before running this cell.
REPO_URL = ""  # e.g. "https://github.com/your-user/jepa2.git"
BRANCH = ""    # optional, e.g. "main"
PROJECT_DIR = Path("/content/jepa2")

if REPO_URL and not (PROJECT_DIR / "model" / "run.py").exists():
    clone_cmd = ["git", "clone"]
    if BRANCH:
        clone_cmd += ["--branch", BRANCH]
    clone_cmd += [REPO_URL, str(PROJECT_DIR)]
    subprocess.run(clone_cmd, check=True)

candidates = [PROJECT_DIR, Path.cwd(), Path("/content/drive/MyDrive/jepa2")]
for candidate in candidates:
    if (candidate / "model" / "run.py").exists():
        PROJECT_DIR = candidate.resolve()
        break
else:
    raise FileNotFoundError(
        "Could not find model/run.py. Set REPO_URL above, or upload the repo to /content/jepa2."
    )

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print(f"Using project: {PROJECT_DIR}")

In [ ]:
import torch

print("python", sys.version)
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. In Colab, choose Runtime > Change runtime type > GPU.")

print("cuda", torch.version.cuda)
print("gpu", torch.cuda.get_device_name(0))
print("gpu memory GB", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
# Colab usually already has CUDA PyTorch installed. Run this cell only if imports fail
# or you intentionally want to change versions, then restart the runtime.
# subprocess.run([sys.executable, "-m", "pip", "install", "-U", "torch", "torchvision", "matplotlib"], check=True)

import matplotlib
print("matplotlib", matplotlib.__version__)

In [ ]:
from model.dataset import download_stl10_if_needed

DATA_ROOT = download_stl10_if_needed(PROJECT_DIR / "datasets" / "stl10" / "stl10_binary")
print(DATA_ROOT)

In [ ]:
# Fast sanity run. This should finish quickly and prove CUDA training works.
smoke_cmd = [
    sys.executable, "-m", "model.run",
    "--data-root", str(DATA_ROOT),
    "--split", "train",
    "--batch-size", "32",
    "--max-steps", "5",
    "--embed-dim", "64",
    "--encoder-depth", "2",
    "--encoder-heads", "4",
    "--num-workers", "2",
    "--device", "cuda",
    "--log-every", "1",
]
subprocess.run(smoke_cmd, check=True, cwd=PROJECT_DIR)

In [ ]:
# Main training configuration. Adjust for your Colab GPU and time budget.
BATCH_SIZE = 128
EPOCHS = 20
MAX_STEPS = None  # set to an int for shorter runs, e.g. 2000
EMBED_DIM = 192
ENCODER_DEPTH = 6
ENCODER_HEADS = 6
OUTPUT_DIR = PROJECT_DIR / "checkpoints"

train_cmd = [
    sys.executable, "-m", "model.run",
    "--data-root", str(DATA_ROOT),
    "--split", "unlabeled",
    "--batch-size", str(BATCH_SIZE),
    "--epochs", str(EPOCHS),
    "--embed-dim", str(EMBED_DIM),
    "--encoder-depth", str(ENCODER_DEPTH),
    "--encoder-heads", str(ENCODER_HEADS),
    "--num-workers", "2",
    "--device", "cuda",
    "--output-dir", str(OUTPUT_DIR),
    "--log-every", "20",
]
if MAX_STEPS is not None:
    train_cmd += ["--max-steps", str(MAX_STEPS)]

print(" ".join(train_cmd))
subprocess.run(train_cmd, check=True, cwd=PROJECT_DIR)

In [ ]:
# Inspect checkpoint keys after training.
ckpt_path = OUTPUT_DIR / "ijepa_latest.pt"
ckpt = torch.load(ckpt_path, map_location="cpu")
print(ckpt_path)
print(sorted(ckpt.keys()))
print("has encoder", "encoder" in ckpt)
print("has predictor", "predictor" in ckpt)
print("has target/context", "target_encoder" in ckpt or "context_encoder" in ckpt)

In [ ]:
# Optional linear probe. Set PROBE_EPOCHS higher for a real evaluation.
RUN_PROBE = True
PROBE_EPOCHS = 10
PROBE_MAX_TRAIN_STEPS = None  # e.g. 100 for a quick probe

if RUN_PROBE:
    probe_cmd = [
        sys.executable, "-m", "model.probe",
        "--checkpoint", str(ckpt_path),
        "--data-root", str(DATA_ROOT),
        "--epochs", str(PROBE_EPOCHS),
        "--batch-size", "512",
        "--probe-batch-size", "512",
        "--num-workers", "2",
        "--device", "cuda",
    ]
    if PROBE_MAX_TRAIN_STEPS is not None:
        probe_cmd += ["--max-train-steps", str(PROBE_MAX_TRAIN_STEPS)]
    print(" ".join(probe_cmd))
    subprocess.run(probe_cmd, check=True, cwd=PROJECT_DIR)

In [ ]:
# Optional: copy outputs to Google Drive.
COPY_TO_DRIVE = False
DRIVE_OUT = Path("/content/drive/MyDrive/jepa2_checkpoints")

if COPY_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    for path in OUTPUT_DIR.glob("*.pt"):
        shutil.copy2(path, DRIVE_OUT / path.name)
        print("copied", path.name)